# D6：Embedding + 向量检索（RAG 的地基）

> 今天做完，你就拥有**「根据意思找内容」**的能力。
> 这是 RAG 里「检索」那一步的全部核心。

**今日产出：**

| 产出 | 说明 |
|---|---|
| 手写余弦相似度 | 不调库，自己算两个向量有多像 |
| `embed_helper.py` | 已经建好，提供 `转向量()` 和 `余弦相似度()` |
| `search()` | 给一个问题 + 一堆文本，返回最相关的 k 条 |
| FAISS 索引 | 几十条用手算，几万条就得用 FAISS |

---

## 第一部分：为什么文本要变成数字

计算机不会「理解」文字，它只会算数字。

那怎么让计算机判断「这两句话意思像不像」？

**笨办法**：比字符串 —— 「今天天气很好」vs「今儿个天气不错」，字面只重合几个字，会被判为不像。

**正确办法**：把每句话变成**一串数字**（向量），然后比较这两串数字的「方向」像不像。

**关键洞察**：意思相近的句子，变出来的向量方向也相近。

## 环境说明：两个已修复的坑（了解即可，不用管）

这两天的系统升级（macOS 26.5 → **27.0.1**）弄坏了两个东西，我已经在 `embed_helper.py` 里绕过：

| 问题 | 表现 | 我怎么修的 |
|---|---|---|
| **scipy 二进制不兼容** | `dlopen(...): section '__DATA/__thread_bss' has a zero-fill section type` | scipy 1.15.3 在 macOS 27 上加载失败，而能修的 scipy 1.16+ 要求 Python 3.11（我们是 3.10）。**但 Embedding 用不到 scipy** —— 它只是 transformers 的无关依赖。所以在 import 前塞了几个空桩模块骗过它 |
| **OpenMP 库冲突** | `OMP: Error #15: Initializing libomp.dylib, but found libomp.dylib already initialized` | torch 和 faiss 各带一份 OpenMP，同时加载就崩。设 `KMP_DUPLICATE_LIB_OK=TRUE` 允许重复加载 |

**你什么都不用做** —— 这些都在 `embed_helper.py` 里处理好了，`from embed_helper import 转向量` 一行就能用。

> **一个工程教训**：环境是会坏的。系统升级、依赖更新都可能让昨天能跑的代码今天跑不了。
> 遇到这种情况，先分清是**代码问题**还是**环境问题** —— 报错里出现 `dlopen`、`.so`、`libomp`、`__thread_bss` 这类词，基本就是环境问题，不是你写错了。


In [1]:
# ============================================================
# 目的：证明「按字面比字符串」反映不了意思
# ============================================================

print("两个字符串能比较'大小'吗？")

# 下面这行：Python 里 < 号对字符串是按「字符编码」逐字比较的
# 能比出结果，但这个结果跟「两句话意思像不像」毫无关系
print('  "今天天气很好" < "今儿个天气不错"  →', "今天天气很好" < "今儿个天气不错")
print("  能，但这是按字符编码比的 —— 跟「意思」毫无关系")
print()   # 空行，只是为了输出好读

a, b = "今天天气很好", "今儿个天气不错"
# 上一行：同时给 a 和 b 赋值（元组解包），等价于 a=... 然后 b=...

# 拆解下面这一行：
#   zip(a, b)                把两个字符串并排拉在一起，一次取出一对字符
#   for x, y in zip(a, b)    每轮拿到一对：x 是 a 的字符，y 是 b 的字符
#   if x == y                这一对字符是否相同
#   sum(1 for ...)           把满足条件的个数加起来
相同字数 = sum(1 for x, y in zip(a, b) if x == y)

# max(len(a), len(b))  取两句话里更长的那一个长度，作为分母
print(f"按字面：{相同字数}/{max(len(a), len(b))} 个字相同")
print("→ 字面上看起来只有一半像")
print()
print("但实际上这两句话意思几乎完全一样。")
print("所以我们需要「按意思比较」的方法 —— 那就是把句子变成向量。")


两个字符串能比较'大小'吗？
  "今天天气很好" < "今儿个天气不错"  → False
  能，但这是按字符编码比的 —— 跟「意思」毫无关系

按字面：1/7 个字相同
→ 字面上看起来只有一半像

但实际上这两句话意思几乎完全一样。
所以我们需要「按意思比较」的方法 —— 那就是把句子变成向量。


---

## 第二部分：手写余弦相似度

### 余弦相似度在算什么？

两个向量，画在坐标系里就是两支箭。

- 两支箭**方向相同** → 相似度 = 1（完全一样）
- 两支箭**垂直** → 相似度 = 0（毫无关系）
- 两支箭**方向相反** → 相似度 = -1（意思相反）

**公式**：两个向量的点积 ÷ 两个向量长度的乘积

用大白话说：**「先看两支箭往一个方向使劲了多少（点积），再除以它们各自的长度，把长度的影响消掉」**。

### 先用 2 维向量手算一遍

2 维好画好算，理解公式之后再上 512 维。

In [2]:
import math   # 导入 math 模块，下面要用 math.sqrt() 开根号


# ============================================================
# 函数 1：点积
#   输入：两个等长的列表 a、b
#   输出：对应位置相乘，再把所有乘积加起来
#   例子：[1, 2] 和 [3, 4] → 1*3 + 2*4 = 11
# ============================================================
def 点积(a, b):
    结果 = 0                       # 累加器：从 0 开始，用来装总和
    for x, y in zip(a, b):         # zip 把 a 和 b 并排拉在一起，一次取出一对
        结果 = 结果 + x * y        # 这一对相乘，然后加进累加器
    return 结果                     # 循环结束后，把总和交出去


# ============================================================
# 函数 2：向量长度
#   就是勾股定理的推广：各分量的平方和，再开根号
#   例子：[3, 4] 的长度 = √(3² + 4²) = √25 = 5
# ============================================================
def 长度(v):
    平方和 = 0                      # 累加器
    for x in v:                     # 遍历向量里的每一个分量
        平方和 = 平方和 + x * x     # 先把分量平方，再加进累加器
    return math.sqrt(平方和)        # 最后开根号


# ============================================================
# 函数 3：余弦相似度
#   公式 = 点积 ÷ (a 的长度 × b 的长度)
#   结果范围：-1（方向相反）~ 0（垂直）~ 1（方向相同）
# ============================================================
def 余弦相似度_手写(a, b):
    return 点积(a, b) / (长度(a) * 长度(b))


# ---------- 下面用 2 维向量演示四种情况 ----------

print("情况 1：两支箭方向相同")
print("  a = [1, 0]   b = [2, 0]")
# 点积 = 1*2 + 0*0 = 2
# |a| = 1，|b| = 2
# 相似度 = 2 / (1*2) = 1.0
print("  相似度 =", round(余弦相似度_手写([1, 0], [2, 0]), 4), " ← 1.0，完全一样（长度不同也算一样）")
print()

print("情况 2：两支箭垂直")
print("  a = [1, 0]   b = [0, 3]")
# 点积 = 1*0 + 0*3 = 0 → 相似度 = 0
print("  相似度 =", round(余弦相似度_手写([1, 0], [0, 3]), 4), " ← 0，毫无关系")
print()

print("情况 3：两支箭方向相反")
print("  a = [1, 2]   b = [-1, -2]")
# 点积 = 1*(-1) + 2*(-2) = -5 → 相似度 = -1
print("  相似度 =", round(余弦相似度_手写([1, 2], [-1, -2]), 4), " ← -1，意思相反")
print()

print("情况 4：方向偏一点")
print("  a = [1, 0]   b = [1, 1]")
# 点积 = 1*1 + 0*1 = 1
# |a| = 1，|b| = √2 ≈ 1.414
# 相似度 = 1 / 1.414 ≈ 0.707
print("  相似度 =", round(余弦相似度_手写([1, 0], [1, 1]), 4), " ← 0.707，有点像")


情况 1：两支箭方向相同
  a = [1, 0]   b = [2, 0]
  相似度 = 1.0  ← 1.0，完全一样（长度不同也算一样）

情况 2：两支箭垂直
  a = [1, 0]   b = [0, 3]
  相似度 = 0.0  ← 0，毫无关系

情况 3：两支箭方向相反
  a = [1, 2]   b = [-1, -2]
  相似度 = -1.0  ← -1，意思相反

情况 4：方向偏一点
  a = [1, 0]   b = [1, 1]
  相似度 = 0.7071  ← 0.707，有点像


---

## 第三部分：真实的 Embedding

刚才用 `[1, 0]` 这种 2 维向量只是演示公式。真实的 Embedding 是 **512 维**。

模型已经配好了：`BAAI/bge-small-zh-v1.5`（中文专用，约 100MB，本地已下载）。

用法就一行：`转向量(["一句话", "另一句话"])`。

> **下载踩过的坑**（记一下，以后自己装模型会遇到）：
> `huggingface.co` 国内连不上；`hf-mirror.com` 能连但速度只有 6KB/s；**魔搭 ModelScope 3.6 秒就下来了**。
> 以后下载模型优先用 ModelScope：`https://www.modelscope.cn/models/作者/模型名`

In [3]:
# 从封装好的工具里导入两个函数：
#   转向量(句子列表)          → 把若干句话变成向量矩阵
#   余弦相似度(向量a, 向量b)  → 返回 -1 ~ 1 之间的一个数字
from embed_helper import 转向量, 余弦相似度

# 准备 4 句话，放进一个列表
句子 = ["今天天气很好", "今儿个天气不错", "我喜欢吃火锅", "检索增强生成能减少幻觉"]

# 一次性把 4 句话全部变成向量
# 返回的是一个矩阵，形状 (4, 512)：4 句话，每句 512 个数字
向量 = 转向量(句子)

print("每句话变成了一串数字：")
# zip(句子, 向量) 把「句子」和「它的向量」并排配对，一次取一对
for s, v in zip(句子, 向量):
    print(f"  {s:22s} → {len(v)} 维")   # len(v) 就是这个向量的维度数（512）
print()

print("前 5 个数字长这样：")
for s, v in zip(句子, 向量):
    # v[:5]          只取前 5 个数字（切片，你之前学过）
    # float(x)       把 torch 的数字转成普通 Python 小数
    # round(..., 3)  保留 3 位小数，否则打印出来一长串
    print(f"  {s:22s} {[round(float(x), 3) for x in v[:5]]}")
print()
print("⚠️ 这些数字单看毫无意义 —— 有意义的是它们之间的「相对关系」")


/Users/pipizhengkai/ai-d1/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1019 16:31:27.177000 15701 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 71/71 [00:00<00:00, 7629.33it/s]

每句话变成了一串数字：
  今天天气很好                 → 512 维
  今儿个天气不错                → 512 维
  我喜欢吃火锅                 → 512 维
  检索增强生成能减少幻觉            → 512 维

前 5 个数字长这样：
  今天天气很好                 [-0.006, 0.043, -0.041, 0.056, 0.04]
  今儿个天气不错                [0.016, 0.045, -0.012, 0.071, -0.013]
  我喜欢吃火锅                 [0.024, -0.042, 0.048, 0.013, -0.03]
  检索增强生成能减少幻觉            [-0.074, 0.016, 0.002, 0.01, 0.019]

⚠️ 这些数字单看毫无意义 —— 有意义的是它们之间的「相对关系」


In [4]:
print("=" * 64)
print("关键验证：两两相似度")
print("=" * 64)

# 外层循环 i：从第 0 句走到第 3 句
for i in range(len(句子)):
    # 内层循环 j：从 i+1 开始
    #   从 i+1 开始有两个好处：
    #     ① 每对只算一次（不算 A-B 又算 B-A）
    #     ② 不会出现自己跟自己比（那个永远是 1.0，没意义）
    for j in range(i + 1, len(句子)):
        # 取第 i 句和第 j 句的向量，算余弦相似度
        s = 余弦相似度(向量[i], 向量[j])
        # {句子[i]:14s}  左对齐、占 14 个字符宽，让输出排整齐
        # {s:.4f}        保留 4 位小数
        print(f"  {句子[i]:14s} ↔ {句子[j]:14s}  {s:.4f}")

print()
print("看第一行：")
print("  「今天天气很好」和「今儿个天气不错」字面只重合几个字，但相似度 0.84 —— 模型理解了意思")
print("  而「今天天气很好」和「检索增强生成」相似度只有 0.28 —— 模型知道它们无关")


关键验证：两两相似度
  今天天气很好         ↔ 今儿个天气不错         0.8441
  今天天气很好         ↔ 我喜欢吃火锅          0.3737
  今天天气很好         ↔ 检索增强生成能减少幻觉     0.2756
  今儿个天气不错        ↔ 我喜欢吃火锅          0.3555
  今儿个天气不错        ↔ 检索增强生成能减少幻觉     0.2934
  我喜欢吃火锅         ↔ 检索增强生成能减少幻觉     0.2046

看第一行：
  「今天天气很好」和「今儿个天气不错」字面只重合几个字，但相似度 0.84 —— 模型理解了意思
  而「今天天气很好」和「检索增强生成」相似度只有 0.28 —— 模型知道它们无关


---

## 第四部分：FAISS —— 向量多了怎么快速找

刚才只有 4 句话，两两比较也就 6 次。

但真实的 RAG 场景：**几万段文档，每段 512 个数字**。

- 暴力比较：每次提问要算几万次余弦相似度 → 慢
- **FAISS**：专门做「快速找最相似的向量」的库，几万条也能毫秒级返回

用法三步：**建索引 → 加向量 → 搜索**。

In [5]:
import faiss                                    # 向量检索库
from embed_helper import 转向量                 # 我们自己封装的转向量函数

# 准备 8 段「文档」—— 假装这是一个小知识库
文档库 = [
    "今天天气很好，适合出门散步",
    "RAG 的核心是先检索再生成",
    "火锅是四川的特色美食",
    "检索增强生成可以减少模型幻觉",
    "明天可能下雨，记得带伞",
    "向量数据库用来做相似度搜索",
    "张三是公司的销售经理",
    "FAISS 是 Facebook 开源的向量检索库",
]

# ① 把所有文档变成向量
#    转向量(文档库)           → torch 的张量
#    .numpy()                 → 转成 numpy 数组
#    .astype("float32")       → FAISS 要求 float32 类型，float64 会报错
文档向量 = 转向量(文档库).numpy().astype("float32")
print("文档向量形状:", 文档向量.shape, "  读作（文档数, 维度）")
# shape 会是 (8, 512)：8 段文档，每段 512 个数字

# ② 建索引
#    IndexFlatL2 的意思是「用欧氏距离，暴力逐一比较」
#    数据量小时用它就够了；数据量大时换成 IndexIVFFlat 之类
索引 = faiss.IndexFlatL2(文档向量.shape[1])   # 参数 = 向量的维度（512）
索引.add(文档向量)                            # 把 8 个向量全部加进索引
print("索引里的向量数量:", 索引.ntotal)        # ntotal = 索引里一共有几个向量

# ③ 搜索
问题 = "怎么减少大模型的幻觉？"
# 注意：问题也要变成向量，而且形状必须是 (1, 512) 的二维数组
# 所以外面套了一层 [问题]
问题向量 = 转向量([问题]).numpy().astype("float32")

# 索引.search(查询向量, 要几条)
# 返回两个数组：
#   距离  形状 (1, 3)  越小越相似（L2 距离）
#   编号  形状 (1, 3)  对应文档在 文档库 里的下标
距离, 编号 = 索引.search(问题向量, 3)

print()
print(f"问题：{问题}")
print("最相关的 3 段：")
for k in range(3):                      # 依次输出第 1、2、3 条结果
    idx = 编号[0][k]                    # [0] 是「第 0 个查询」（我们只查了 1 个），[k] 是第 k 条
    print(f"  {k+1}. [{距离[0][k]:.3f}] {文档库[idx]}")


文档向量形状: (8, 512)   读作（文档数, 维度）
索引里的向量数量: 8

问题：怎么减少大模型的幻觉？
最相关的 3 段：
  1. [0.531] 检索增强生成可以减少模型幻觉
  2. [1.379] RAG 的核心是先检索再生成
  3. [1.396] FAISS 是 Facebook 开源的向量检索库


---

## 第五部分：任务 —— 自己写一个 `search()`

FAISS 虽好，但**先手写一遍暴力版**，你才会真正明白检索在干什么。

**要求**：给一个问题、一堆文档、要几条，返回最相关的 k 条文档文字。

**例子：**

```
search("怎么减少幻觉", 文档库, 3)  →  返回 3 条最相关的文档文字
```

**骨架：**

```python
def search(问题, 文档列表, k=3):
    # ① 把问题变成向量
    问题向量 = 转向量([问题])[0]

    # ② 把所有文档变成向量
    文档向量 = 转向量(文档列表)

    # ③ 算每条文档的相似度，存成「(相似度, 下标)」的对
    打分的 = []
    for i in range(len(文档列表)):
        分数 = 余弦相似度(问题向量, 文档向量[i])
        打分的.append((分数, i))

    # ④ 按分数从大到小排序  ← 昨天学过 sorted + lambda
    打分的 = sorted(打分的, key=lambda 对: 对[0], reverse=True)

    # ⑤ 取前 k 个，把对应文档文字装进结果列表
    结果 = []
    for 对 in 打分的[:k]:
        结果.append(文档列表[对[1]])
    return 结果
```

**第 ④ 步用的 `sorted` + `lambda`，就是你前两天写过的那个。** 只是这次排序的是「分数和下标」。

**第 ⑤ 步的 `对[1]`** —— `对` 是 `(分数, 下标)` 这个小包裹，`对[0]` 是分数，`对[1]` 是下标。这就是你 D2 学的「for 拆包」。

In [6]:
# ============================================================
# 任务：自己写一个暴力版检索函数
# ============================================================
def search(问题, 文档列表, k=3):
    问题向量 = 转向量([问题])[0]              # 问题 → 512 维向量
    文档向量 = 转向量(文档列表)                # 所有文档 → (篇数, 512) 的矩阵

    打分的 = []                                # 装 (分数, 下标) 的包裹
    for i in range(len(文档列表)):              # 一篇一篇来
        分数 = 余弦相似度(问题向量, 文档向量[i])   # 问题 ↔ 第 i 篇，有多像
        打分的.append((分数, i))                  # 把分数和下标打包收起来
    打分的 = sorted(打分的, key=lambda 对: 对[0], reverse=True)  # 按分数从大到小

    结果 = []
    for 对 in 打分的[:k]:                       # 取前 k 个包裹
        结果.append(文档列表[对[1]])              # 对[1] 是下标 → 换回正文
    return 结果


# ---------------- 自测（这一段不能删！）----------------
结果 = search("怎么减少大模型的幻觉？", 文档库, 3)
print("问题：怎么减少大模型的幻觉？")
for i, d in enumerate(结果, 1):
    print(f"  {i}. {d}")
print()
assert any("幻觉" in d for d in 结果), "最相关的里面应该出现幻觉相关的文档"
assert len(结果) == 3
print("D6 任务通过 ✅")

问题：怎么减少大模型的幻觉？
  1. 检索增强生成可以减少模型幻觉
  2. RAG 的核心是先检索再生成
  3. FAISS 是 Facebook 开源的向量检索库

D6 任务通过 ✅


---

## 收尾

### 今天你应该能回答

1. 为什么文本要变成向量？（计算机只会算数字，不会比意思）
2. 余弦相似度在算什么？（两支箭方向有多像，除以长度消掉长度影响）
3. 意思相近的句子，向量有什么特点？（方向相近，相似度接近 1）
4. FAISS 是干什么的？（向量多了以后，快速找最相似的几个）

### 和第 2 天那个「关键词检索」的区别

你 D2 写过 `count_keywords` —— 那是**按字面匹配**，只能找「包含这个词」的文档。

**Embedding 检索是按意思匹配** —— 你问「怎么减少幻觉」，它能找到「检索增强生成可以减少模型幻觉」这种意思相关的文档，即使字面不完全一样。

**这就是为什么 RAG 离不开 Embedding。**

### 一个必须知道的限制

模型能理解的语义是有限的，小模型在专业领域会失准。

所以真实 RAG 里很少只靠向量检索，通常是：
- **向量检索**（按意思找）+ **关键词检索**（按字面找）→ 混合

这正好是你 D2 写的 `count_keywords` 和今天的 `转向量` 的结合。**D9 做 RAG 优化时会用到。**

---

## 明天预告：D7（10/19）

**文档加载与切分** —— 开始搭真正的 RAG 项目第 1 步。

把一份真实 PDF 读进来，切成一段一段。切分策略直接影响检索效果，是 RAG 里最容易被低估的一步。
